# Amazon ML Challenge 2026 — Business Entity Resolution (Kaggle runner)

Runs the full pipeline from https://github.com/Karanpl-1945/amazon_ml_challenge on Kaggle.

**Before running:** add the challenge data as an input dataset (right panel → *Add Input*),
set **Accelerator = None** and **Internet = On** (Settings panel).
Use **Save Version → Save & Run All** so it keeps running in the background (full run ≈ 2.5–3.5 h).

## 1. Get the code and the two missing libraries

In [ ]:
import os, glob, subprocess, sys
REPO = "/kaggle/working/amazon_ml_challenge"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Karanpl-1945/amazon_ml_challenge.git {REPO}
!pip install -q rapidfuzz anyascii
PKG = f"{REPO}/6a7c763e089e4_aic_2026/student_resource/code/business_entity_resolution"
import numpy, pandas, pyarrow, numba, lightgbm, rapidfuzz
print(sys.version)
print("numpy", numpy.__version__, "| pyarrow", pyarrow.__version__, "| numba", numba.__version__,
      "| lightgbm", lightgbm.__version__, "| rapidfuzz", rapidfuzz.__version__)
print("CPUs:", os.cpu_count())

## 2. Locate the uploaded data (the folder that contains `train/` and `test/`)

In [ ]:
hits = glob.glob("/kaggle/input/**/train_source1.tsv", recursive=True)
assert hits, "train_source1.tsv not found - did you add the challenge dataset as input?"
DATA = os.path.dirname(os.path.dirname(hits[0]))
print("DATA =", DATA)
for sub in ("train", "test"):
    print(sub, sorted(os.listdir(f"{DATA}/{sub}")))
WORKERS = os.cpu_count() or 4

def run(cmd, data, work, out):
    env = dict(os.environ, ER_DATA_DIR=data, ER_WORK_DIR=work, ER_OUTPUT_DIR=out, PYTHONIOENCODING="utf-8")
    p = subprocess.Popen(cmd, cwd=PKG, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    assert p.returncode == 0, f"FAILED (exit {p.returncode}) - see the log above"

## 3. Smoke test on a 0.5% sample (~3 min)
Checks the environment end to end. It must finish with `PASS — no blocking issues found`.

In [ ]:
SMP = "/kaggle/working/sample"
run([sys.executable, "src/make_sample.py", "--out", SMP, "--frac", "0.005"], DATA, "/kaggle/working/work", "/kaggle/working/output")
run([sys.executable, "src/run_pipeline.py", "--workers", str(WORKERS)], SMP, f"{SMP}/work", f"{SMP}/output")

## 4. Full run (≈ 2.5–3.5 h on 4 CPUs)
If it stops part-way, re-run this cell with `"--from-step", "N"` added (N = the failed step) — earlier results are reused.

In [ ]:
!rm -rf /kaggle/working/sample
run([sys.executable, "src/run_pipeline.py", "--workers", str(WORKERS)],
    DATA, "/kaggle/working/work", "/kaggle/working/output")

## 5. Results — download from the *Output* tab (`output/` files and `results.zip`)

In [ ]:
import json
d = json.load(open("/kaggle/working/work/model/decision.json"))
print(f"validation macro F0.5 = {d['f05']:.4f} | threshold {d['threshold']} | assign_best {d['assign_best']} | "
      f"blocking ceiling {d['blocking_ceiling_f05']:.4f}")
!cp /kaggle/working/work/pipeline_log.txt /kaggle/working/work/model/decision.json /kaggle/working/output/
!cd /kaggle/working && zip -q -r results.zip output
!ls -la /kaggle/working/output
!head -5 /kaggle/working/output/matching_results.tsv

Optional clean-up so the saved version stays small (`work/` is ~8 GB of intermediate files):

In [ ]:
# !rm -rf /kaggle/working/work/clean /kaggle/working/work/cand /kaggle/working/work/feat